<a href="https://colab.research.google.com/github/Mouad-Tazka/-ds2002-fa26/blob/main/notebooks/05-etl-walmart/2026-10-16%20%E2%80%94%20Walmart%20Warm-Up%20SKU%20Mess%20%E2%80%94%20Lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DS2002 · Walmart Warm-Up SKU Mess

**Lab — 2026-10-16 · Fall 2026**  

---

## Lab 08 — Walmart Warm-Up (SKU Mess)

This lab uses the **real midterm data** in `../06-midterm-walmart/data/`. It's a warm-up so you hit the ground running on the project. On Kaggle/Colab, upload the data folder or adjust the path.

*If the file isn't found, the cell falls back to a small inline sample so the notebook still runs.*

In [8]:
import pandas as pd

PATH = '/content/product_catalog.csv'

cat = pd.read_csv(PATH)

print("Dataset shape:", cat.shape)
cat.head()

Dataset shape: (45, 7)


,sku,product_name,category,subcategory,unit_cost,supplier,package_size
0,PT-12,pop-tarts strawberry,Breakfast Foods,NaN,1.93,Johnson & Johnson,4-pack
1,SKU#459812,Strawberry Pop-Tart,Breakfast,Pop-Tarts,2.26,Roku Inc,4pk
2,POPTART-STRAW,Pop-Tarts Strawberry,Breakfast,NaN,2.11,Sara Lee,6 pack
3,PT_STRAWBERRY,pop-tarts strawberry,breakfast,Pop-Tarts,NaN,Campbell's,4 pack
4,WTR-24,BOTTLED WATER 24PK,Emergency Supplies,Bottled,2.64,Sara Lee,1 pack


### Q1 — How many rows look like Pop-Tart products?

Filter on `product_name` containing `pop`, case-insensitively. Print the count and the matching rows.

In [11]:
# TODO (hint: .str.lower().str.contains('pop'))
pop_tarts = cat[
    cat["product_name"].str.lower().str.contains("pop", case=False, na=False)
]

print("Number of Pop-Tart rows:", len(pop_tarts))
display(pop_tarts)

Number of Pop-Tart rows: 4


,sku,product_name,category,subcategory,unit_cost,supplier,package_size
0,PT-12,pop-tarts strawberry,Breakfast Foods,NaN,1.93,Johnson & Johnson,4-pack
1,SKU#459812,Strawberry Pop-Tart,Breakfast,Pop-Tarts,2.26,Roku Inc,4pk
2,POPTART-STRAW,Pop-Tarts Strawberry,Breakfast,NaN,2.11,Sara Lee,6 pack
3,PT_STRAWBERRY,pop-tarts strawberry,breakfast,Pop-Tarts,NaN,Campbell's,4 pack


### Q2 — Which SKUs and names all mean *strawberry* Pop-Tart?

List them. Then note which Pop-Tart rows are **not** strawberry — those must stay separate, and saying so is part of the answer.

In [12]:
strawberry_pop_tarts = pop_tarts[
    pop_tarts["product_name"].str.contains(
        "strawberry", case=False, na=False
    )
]

other_pop_tarts = pop_tarts[
    ~pop_tarts["product_name"].str.contains(
        "strawberry", case=False, na=False
    )
]

print("Strawberry Pop-Tart products:")
display(strawberry_pop_tarts[["sku", "product_name", "package_size"]])

print("Non-strawberry Pop-Tart products:")
display(other_pop_tarts)

Strawberry Pop-Tart products:


,sku,product_name,package_size
0,PT-12,pop-tarts strawberry,4-pack
1,SKU#459812,Strawberry Pop-Tart,4pk
2,POPTART-STRAW,Pop-Tarts Strawberry,6 pack
3,PT_STRAWBERRY,pop-tarts strawberry,4 pack


Non-strawberry Pop-Tart products:


,sku,product_name,category,subcategory,unit_cost,supplier,package_size


**Pop-Tart rows that are a different product:** _..._

### Q3 — Build the mapping

A dict from each variant SKU to one canonical SKU. Then apply it in a way that leaves unmapped SKUs as themselves rather than turning them into `NaN` — that single mistake silently deletes most of the catalog.

In [13]:
# Keep track of the original number of rows.
original_row_count = len(cat)

# Map the different strawberry Pop-Tart SKUs to one SKU.
sku_map = {
    "SKU#459812": "PT-12",
    "POPTART-STRAW": "PT-12",
    "PT_STRAWBERRY": "PT-12"
}

cat["canonical_sku"] = cat["sku"].map(sku_map).fillna(cat["sku"])

display(
    cat[cat["product_name"].str.contains(
        "pop", case=False, na=False
    )][["sku", "product_name", "canonical_sku"]]
)

print("Original unique SKUs:", cat["sku"].nunique())
print("Canonical unique SKUs:", cat["canonical_sku"].nunique())


,sku,product_name,canonical_sku
0,PT-12,pop-tarts strawberry,PT-12
1,SKU#459812,Strawberry Pop-Tart,PT-12
2,POPTART-STRAW,Pop-Tarts Strawberry,PT-12
3,PT_STRAWBERRY,pop-tarts strawberry,PT-12


Original unique SKUs: 45
Canonical unique SKUs: 42


### Q4 — Normalize `category`

Strip and title-case, then show the distinct values before and after with counts. Note any pair that is still two labels for one thing — `Breakfast` and `Breakfast Foods` are not the same string but they are the same category.

In [14]:
# Count the original category names.
print("Categories before cleaning:")
print(cat["category"].value_counts(dropna=False))

# Clean up capitalization and extra spaces.
cat["category_clean"] = cat["category"].str.strip().str.title()

# Count the cleaned category names.
print("\nCategories after cleaning:")
print(cat["category_clean"].value_counts(dropna=False))

print("\nUnique categories before:", cat["category"].nunique())
print("Unique categories after:", cat["category_clean"].nunique())

Categories before cleaning:
category
Emergency              5
Emergency Supplies     3
EMERGENCY              3
ENTERTAINMENT          3
GROCERY                3
Breakfast              2
entertainment          2
household              2
ELECTRONICS            2
health                 2
Baby                   2
Electronics            1
ALCOHOL                1
Alcoholic Beverages    1
breakfast              1
Breakfast Foods        1
Groceries              1
Grocery                1
electronics            1
snacks                 1
grocery                1
HOUSEHOLD              1
Snacks                 1
Household              1
baby                   1
Baby Products          1
emergency              1
Name: count, dtype: int64

Categories after cleaning:
category_clean
Emergency              9
Entertainment          5
Grocery                5
Electronics            4
Household              4
Emergency Supplies     3
Breakfast              3
Baby                   3
Snacks             

After cleaning the category column, the number of unique categories dropped from 27 to 15. This was mostly because some categories had different capitalization or extra spaces. However, I noticed that some categories still mean the same thing even though they're written differently. For example, "Breakfast" and "Breakfast Foods" are still separate labels. I would probably group those into one category to make the data more consistent.

### Q5 — Find the fragmentation you have not spotted yet

Hand-scanning a catalog does not scale. **TODO:** build a candidate list programmatically — normalize each product name (lowercase, strip punctuation, sort the words) and group by that, then show any group holding more than one SKU.

This is the workflow from Monday's lecture, and it is how you will actually approach the full catalog on the midterm.

In [ ]:
import re

def normalize_name(name):
    words = re.findall(r"[a-z0-9]+", str(name).lower())
    return " ".join(sorted(words))

cat["name_key"] = cat["product_name"].apply(normalize_name)

sku_counts = cat.groupby("name_key")["sku"].nunique()

duplicate_names = sku_counts[sku_counts > 1].index

candidates = cat[
    cat["name_key"].isin(duplicate_names)
].sort_values("name_key")

print("Potential duplicate groups:", len(duplicate_names))
print("Rows involved:", len(candidates))

display(candidates[
    ["sku", "product_name", "name_key", "package_size"]
])

### Q6 — Prove the mapping is safe

**TODO:** assert that applying your mapping produced no nulls in the canonical column and did not change the number of rows. Consolidation renames; it never adds or removes.

In [16]:
assert cat["canonical_sku"].notna().all(), \
    "There are missing canonical SKUs!"

# Make sure the number of rows stayed the same.
assert len(cat) == original_row_count, \
    "The number of rows changed!"

unmapped = ~cat["sku"].isin(sku_map)

assert (
    cat.loc[unmapped, "canonical_sku"]
    == cat.loc[unmapped, "sku"]
).all(), "Some unmapped SKUs were changed!"

print("All checks passed!")
print("Original rows:", original_row_count)
print("Rows after mapping:", len(cat))
print("Missing canonical SKUs:", cat["canonical_sku"].isna().sum())

All checks passed!
Original rows: 45
Rows after mapping: 45
Missing canonical SKUs: 0


All of my assertions passed. The catalog still has 45 rows after applying the mapping, and there are no missing values in the canonical SKU column. I also checked that products not included in my mapping kept their original SKUs. This gives me more confidence that the mapping worked as intended without accidentally removing or changing unrelated products.

### Write-up

**a)** Name two product families beyond Pop-Tarts that appear fragmented, and how you would group them.

**b)** Where does consolidation stop being a data-cleaning task and become a business decision? Give one example from this catalog where you genuinely are not sure, and say who you would ask.

**a)** One example I found was canned soup. The catalog has two different SKUs, `SOUP-CAN` and `CNSOUP-01`, even though the product names are basically the same thing, just with the words in a different order. Both are also listed as single items. I would group them as potential duplicates and check whether they are actually the same product before giving them one canonical SKU.

Another example is Monopoly. I noticed three different SKUs (`BG-MONO`, `MONOPOLY-01`, and `GAME-MONO`) that all describe a Monopoly board game. I would group these together for review since they seem to refer to the same game. However, one of them has a different package size, so I would want to confirm that before officially combining them.


**b)** I think consolidation becomes more of a business decision when two products look similar in the data but might actually be different items. Fixing capitalization or removing punctuation is pretty straightforward, but deciding whether two SKUs represent the exact same product takes more information.

One example I wasn't completely sure about was the strawberry Pop-Tarts. All four entries have basically the same product name, so I initially thought they should all be combined. However, one of them is listed as a 6-pack while the others are 4-packs. If those package sizes are correct, they probably shouldn't all share the same inventory SKU.

I would ask someone on the inventory or merchandising team to confirm whether those products are actually identical. Overall, I think data cleaning can help identify possible duplicates, but the final decision should depend on the actual products rather than just their names.